Регрессия: предсказание IC50
Сравнение моделей: LinearRegression, Ridge, Lasso, ElasticNet,
                   RandomForest, GradientBoosting.
Метрика: RMSE, MAE, R² (на log1p(IC50)).

In [2]:
import os
os.makedirs("plots", exist_ok=True)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import VarianceThreshold
import warnings

warnings.filterwarnings("ignore")

## Данные

In [3]:
df = pd.read_excel("data/data.xlsx", index_col=0)
targets = ["IC50, mM", "CC50, mM", "SI"]
features = [c for c in df.columns if c not in targets]

X = df[features]
y = np.log1p(df["IC50, mM"])  # log-трансформация для нормализации

# Удаляем признаки с нулевой дисперсией
selector = VarianceThreshold(threshold=0.0)
X = X.fillna(X.median())
X_sel = selector.fit_transform(X)
sel_features = [features[i] for i in selector.get_support(indices=True)]
X = pd.DataFrame(X_sel, columns=sel_features)
print(f"Признаков после фильтрации: {X.shape[1]}")

cv = KFold(n_splits=5, shuffle=True, random_state=42)


def cv_metrics(model, X, y, cv):
    """Возвращает среднее RMSE, MAE, R² по кросс-валидации."""
    rmse_scores = -cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    mae_scores  = -cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error")
    r2_scores   =  cross_val_score(model, X, y, cv=cv, scoring="r2")
    return rmse_scores.mean(), mae_scores.mean(), r2_scores.mean()

Признаков после фильтрации: 192


## Базовые модели

In [4]:
print("\n── Базовые модели (дефолтные параметры) ──")

base_models = {
    "Ridge":            Pipeline([("sc", StandardScaler()), ("m", Ridge())]),
    "Lasso":            Pipeline([("sc", StandardScaler()), ("m", Lasso(max_iter=5000))]),
    "ElasticNet":       Pipeline([("sc", StandardScaler()), ("m", ElasticNet(max_iter=5000))]),
    "RandomForest":     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in base_models.items():
    rmse, mae, r2 = cv_metrics(model, X, y, cv)
    results[name] = {"RMSE": rmse, "MAE": mae, "R²": r2}
    print(f"  {name:22s}: RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")


── Базовые модели (дефолтные параметры) ──
  Ridge                 : RMSE=2.2123  MAE=1.3620  R²=-0.7623
  Lasso                 : RMSE=1.8596  MAE=1.5427  R²=-0.0038
  ElasticNet            : RMSE=1.8567  MAE=1.5407  R²=-0.0006
  RandomForest          : RMSE=1.4199  MAE=1.1158  R²=0.4112
  GradientBoosting      : RMSE=1.4272  MAE=1.1428  R²=0.4063


## Подбор гиперпараметров

In [5]:
print("\n── GridSearchCV: Ridge ──")
ridge_pipe = Pipeline([("sc", StandardScaler()), ("m", Ridge())])
ridge_params = {"m__alpha": [0.01, 0.1, 1, 10, 100, 500]}
gs_ridge = GridSearchCV(ridge_pipe, ridge_params, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
gs_ridge.fit(X, y)
print(f"  Лучший alpha: {gs_ridge.best_params_}")
print(f"  CV RMSE: {-gs_ridge.best_score_:.4f}")

print("\n── GridSearchCV: Lasso ──")
lasso_pipe = Pipeline([("sc", StandardScaler()), ("m", Lasso(max_iter=10000))])
lasso_params = {"m__alpha": [0.001, 0.01, 0.1, 0.5, 1]}
gs_lasso = GridSearchCV(lasso_pipe, lasso_params, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
gs_lasso.fit(X, y)
print(f"  Лучший alpha: {gs_lasso.best_params_}")
print(f"  CV RMSE: {-gs_lasso.best_score_:.4f}")

print("\n── GridSearchCV: RandomForest ──")
rf_params = {
    "n_estimators": [100, 200],
    "max_depth":    [None, 10, 20],
    "min_samples_split": [2, 5],
}
gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_rf.fit(X, y)
print(f"  Лучшие параметры: {gs_rf.best_params_}")
print(f"  CV RMSE: {-gs_rf.best_score_:.4f}")

print("\n── GridSearchCV: GradientBoosting ──")
gb_params = {
    "n_estimators":   [100, 200],
    "learning_rate":  [0.05, 0.1, 0.2],
    "max_depth":      [3, 5],
}
gs_gb = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_params, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_gb.fit(X, y)
print(f"  Лучшие параметры: {gs_gb.best_params_}")
print(f"  CV RMSE: {-gs_gb.best_score_:.4f}")


── GridSearchCV: Ridge ──
  Лучший alpha: {'m__alpha': 500}
  CV RMSE: 1.9105

── GridSearchCV: Lasso ──
  Лучший alpha: {'m__alpha': 0.1}
  CV RMSE: 1.6566

── GridSearchCV: RandomForest ──
  Лучшие параметры: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
  CV RMSE: 1.4028

── GridSearchCV: GradientBoosting ──
  Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}
  CV RMSE: 1.4210


## Сводная таблица

In [6]:
tuned = {
    "Ridge (tuned)":            -gs_ridge.best_score_,
    "Lasso (tuned)":            -gs_lasso.best_score_,
    "RandomForest (tuned)":     -gs_rf.best_score_,
    "GradientBoosting (tuned)": -gs_gb.best_score_,
}

print("\n── Итоговое сравнение RMSE (CV, log-шкала) ──")
all_rmse = {k: v["RMSE"] for k, v in results.items()}
all_rmse.update(tuned)
for name, rmse in sorted(all_rmse.items(), key=lambda x: x[1]):
    print(f"  {name:30s}: RMSE = {rmse:.4f}")


── Итоговое сравнение RMSE (CV, log-шкала) ──
  RandomForest (tuned)          : RMSE = 1.4028
  RandomForest                  : RMSE = 1.4199
  GradientBoosting (tuned)      : RMSE = 1.4210
  GradientBoosting              : RMSE = 1.4272
  Lasso (tuned)                 : RMSE = 1.6566
  ElasticNet                    : RMSE = 1.8567
  Lasso                         : RMSE = 1.8596
  Ridge (tuned)                 : RMSE = 1.9105
  Ridge                         : RMSE = 2.2123


## Feature Importance (лучшая модель RF)

In [7]:
best_rf = gs_rf.best_estimator_
importances = best_rf.feature_importances_
top_idx = np.argsort(importances)[-20:][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([X.columns[i] for i in top_idx[::-1]], importances[top_idx[::-1]], color="steelblue")
ax.set_title("RandomForest: Топ-20 важных признаков (IC50)")
ax.set_xlabel("Feature Importance")
plt.tight_layout()
plt.savefig("plots/ic50_feature_importance.png")
plt.close()
print("\nСохранён график: ic50_feature_importance.png")


Сохранён график: ic50_feature_importance.png


## Предсказания vs Факт

In [8]:
y_pred = gs_rf.predict(X)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y, y_pred, alpha=0.3, s=10, color="steelblue")
mn, mx = y.min(), y.max()
ax.plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Идеальное предсказание")
ax.set_xlabel("log1p(IC50) — факт")
ax.set_ylabel("log1p(IC50) — предсказание")
ax.set_title(f"RandomForest (tuned): R²={r2_score(y, y_pred):.3f}")
ax.legend()
plt.tight_layout()
plt.savefig("plots/ic50_pred_vs_actual.png")
plt.close()
print("Сохранён график: ic50_pred_vs_actual.png")

Сохранён график: ic50_pred_vs_actual.png


## Итоговые выводы

- После удаления признаков с нулевой дисперсией осталось 192 дескриптора.
- Линейные модели (Ridge, Lasso, ElasticNet) показывают плохое качество: R² близок к нулю или отрицательный, что говорит о неспособности объяснить дисперсию целевой переменной.
- Lasso после настройки (alpha=0.1) дает RMSE=1.6566 — лучший результат среди линейных.
- Деревянные модели (Random Forest и GradientBoosting) значительно лучше: R² достигает 0.41, RMSE около 1.42.
- Настройка гиперпараметров улучшает Random Forest до RMSE=1.4028 (max_depth=10, min_samples_split=5, 200 деревьев).
- GradientBoosting после настройки (learning_rate=0.05, max_depth=3, 200 деревьев) даёт RMSE=1.4210, что близко к Random Forest.
- Лучшая модель — Random Forest с подобранными параметрами (RMSE=1.4028).
- Разрыв между линейными и древовидными моделями большой (~0.25 по RMSE), что указывает на нелинейные зависимости в данных.
- Качество остается умеренным (R²=0.41) — предсказывать селективность (SI) по молекулярным дескрипторам сложно.